# Overview

# parsing: include all headings (h2 elementes in html file)

In [9]:
%pip install lxml



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install docling

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 13.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 12.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 13.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 12.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 13.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 13.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 MB 13.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 13.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 13.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━

In [7]:
# Imports
from bs4 import BeautifulSoup
import pandas as pd
import os
import matplotlib.pyplot as plt
from docling.document_converter import DocumentConverter
import re

/opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Read

In [ ]:
import os
import glob
from pathlib import Path

def read_all_html_files(base_folder):
    """
    Read all HTML files from the HKNews directory structure
    Structure: HKNews -> category_folders -> year_folders -> month_folders -> *.html
    
    Returns:
        dict: {filename: html_content} for all HTML files found
    """
    html_files_content = {}
    
    # Use glob to find all .html files recursively
    pattern = os.path.join(base_folder, '**', '*.html')
    html_files = glob.glob(pattern, recursive=True)
    
    print(f"Found {len(html_files)} HTML files")
    
    for file_path in html_files:
        try:
            # Read the HTML content
            with open(file_path, 'r', encoding='utf-8') as file:
                content = file.read()
            
            # Use just the filename as key (you can modify this if needed)
            filename = os.path.basename(file_path)
            
            # Handle duplicate filenames by adding path info
            if filename in html_files_content:
                # Create unique key using parent directories
                path_parts = Path(file_path).parts
                if 'HKNews' in path_parts:
                    hknews_index = path_parts.index('HKNews')
                    # Use category/year/month/filename as unique identifier
                    unique_parts = path_parts[hknews_index+1:]
                    filename = '_'.join(unique_parts)
            
            html_files_content[filename] = content
            
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    
    return html_files_content

def get_html_files_with_metadata(base_folder):
    """
    Read all HTML files and include metadata about their location
    
    Returns:
        dict: {filename: {'content': html_content, 'metadata': {...}}}
    """
    html_files_data = {}
    
    pattern = os.path.join(base_folder, '**', '*.html')
    html_files = glob.glob(pattern, recursive=True)
    
    print(f"Found {len(html_files)} HTML files")
    
    for file_path in html_files:
        try:
            # Read the HTML content
            with open(file_path, 'r', encoding='utf-8') as file:
                content = file.read()
            
            # Extract path information
            path_parts = Path(file_path).parts
            filename = os.path.basename(file_path)
            
            # Parse directory structure
            metadata = {
                'full_path': file_path,
                'category': 'unknown',
                'year': 'unknown', 
                'month': 'unknown',
                'file_size': os.path.getsize(file_path)
            }
            
            if 'HKNews' in path_parts:
                hknews_index = path_parts.index('HKNews')
                if len(path_parts) > hknews_index + 1:
                    metadata['category'] = path_parts[hknews_index + 1]
                if len(path_parts) > hknews_index + 2:
                    metadata['year'] = path_parts[hknews_index + 2]
                if len(path_parts) > hknews_index + 3:
                    metadata['month'] = path_parts[hknews_index + 3]
            
            # Create unique filename if needed
            unique_filename = filename
            if unique_filename in html_files_data:
                unique_filename = f"{metadata['category']}_{metadata['year']}_{metadata['month']}_{filename}"
            
            html_files_data[unique_filename] = {
                'content': content,
                'metadata': metadata
            }
            
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    
    return html_files_data

def preview_html_files(base_folder, max_preview=5):
    """
    Preview the first few HTML files found in the directory
    """
    pattern = os.path.join(base_folder, '**', '*.html')
    html_files = glob.glob(pattern, recursive=True)
    
    print(f"Total HTML files found: {len(html_files)}")
    print("\nFile structure preview:")
    
    for i, file_path in enumerate(html_files[:max_preview]):
        path_parts = Path(file_path).parts
        if 'HKNews' in path_parts:
            hknews_index = path_parts.index('HKNews')
            relative_path = '/'.join(path_parts[hknews_index:])
            print(f"{i+1}. {relative_path}")
            
            # Show file size
            size_kb = os.path.getsize(file_path) / 1024
            print(f"   Size: {size_kb:.1f} KB")
    
    if len(html_files) > max_preview:
        print(f"   ... and {len(html_files) - max_preview} more files")

# Main execution
if __name__ == "__main__":
    base_folder = '/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/HKNews'
    
    # Preview the files first
    print("=== HTML Files Preview ===")
    preview_html_files(base_folder)
    
    # Method 1: Simple dictionary with filename and content
    print("\n=== Reading HTML Files (Simple) ===")
    examples = read_all_html_files(base_folder)
    print(f"Successfully read {len(examples)} HTML files")
    
    # Method 2: With metadata (uncomment if you need metadata)
    # print("\n=== Reading HTML Files (With Metadata) ===")
    # html_data_with_metadata = get_html_files_with_metadata(base_folder)
    # print(f"Successfully read {len(html_data_with_metadata)} HTML files with metadata")
    
    # Show some statistics
    if examples:
        print("\nContent statistics:")
        content_lengths = [len(content) for content in examples.values()]
        print(f"Average file size: {sum(content_lengths) / len(content_lengths):.0f} characters")
        print(f"Largest file: {max(content_lengths)} characters") 
        print(f"Smallest file: {min(content_lengths)} characters")
        
        # Show first few filenames
        print(f"\nFirst few filenames:")
        for i, filename in enumerate(list(examples.keys())[:3]):
            print(f"  {i+1}. {filename}")

# Example usage for your hybrid parser:
html_files = read_all_html_files(base_folder)
# 
# hybrid_results = []
# for filename, html_content in examples.items():
#     result = hybrid_parser_from_content(filename, html_content)
#     hybrid_results.append(result)

## parsing script (from 02_hybrid_parsing)

In [6]:
import os
from docling.document_converter import DocumentConverter
from pathlib import Path

def parse_with_bs4(html_content):
    from bs4 import BeautifulSoup

    soup = BeautifulSoup(html_content, 'html.parser')

    # Remove script and style elements
    for script_or_style in soup(["script", "style"]):
        script_or_style.decompose()

    # Identify main article sections
    sections = soup.find_all('div', class_='text-image cq-dd-image')
    
    all_structured_content = []  # List of {title, paragraphs} dictionaries
    footer_keywords = ['newsletter', 'staffnet', 'globe', 'download']
    
    for section in sections:
        # Process figures and captions first
        captions = []
        for fig in section.find_all('figure'):
            figcaption = fig.find('figcaption')
            if figcaption:
                caption_text = figcaption.get_text(" ", strip=True)
                if caption_text:
                    captions.append(caption_text)
            fig.decompose()

        # Find all h2 elements and their following content within this section
        elements = section.find_all(['h2', 'p'])
        
        current_title = None
        current_paragraphs = []
        
        # Add captions to the first content block
        if captions:
            current_paragraphs.extend(captions)
        
        for element in elements:
            if element.name == 'h2':
                # Save previous title-paragraph group if it exists
                if current_title is not None or current_paragraphs:
                    title_to_use = current_title if current_title else "Main article"
                    # Skip if it's a footer section
                    if not any(kw in title_to_use.lower() for kw in footer_keywords):
                        all_structured_content.append({
                            'title': title_to_use,
                            'paragraphs': current_paragraphs.copy()
                        })
                
                # Start new title-paragraph group
                current_title = element.get_text(strip=True)
                current_paragraphs = []
                
            elif element.name == 'p':
                text = element.get_text(" ", strip=True)
                # Clean up link artifacts
                text = text.replace("external page", "").replace("call_made", "")
                text = ' '.join(text.split())  # Normalize extra spaces
                
                # Filter out short or footer content
                if (text and len(text) >= 20 and 
                    not any(kw in text.lower() for kw in footer_keywords)):
                    current_paragraphs.append(text)
        
        # Don't forget the last group
        if current_title is not None or current_paragraphs:
            title_to_use = current_title if current_title else "Main article"
            if not any(kw in title_to_use.lower() for kw in footer_keywords):
                all_structured_content.append({
                    'title': title_to_use,
                    'paragraphs': current_paragraphs.copy()
                })
    
    # If no structured content was found, create a default entry
    if not all_structured_content:
        all_structured_content.append({
            'title': "Main article",
            'paragraphs': []
        })
    
    # Extract all titles and paragraphs for backward compatibility
    all_titles = [content['title'] for content in all_structured_content]
    all_paragraphs = []
    for content in all_structured_content:
        all_paragraphs.extend(content['paragraphs'])
    
    return {
        'titles': all_titles,
        'body': '<br><br>'.join(all_paragraphs),
        'paragraphs': all_paragraphs,
        'structured_content': all_structured_content  # New: maintain title-paragraph relationships
    }

# Create a Docling converter
docling_converter = DocumentConverter()

def parse_with_docling(filepath): 
    result = docling_converter.convert(filepath)
    text = result.document.export_to_markdown()
    return {
        'body': text
    }

def inject_docling_bullets(bs4_body: str, docling_markdown: str) -> str:
    # Extract bullet-style lines from docling
    bullet_lines = [
        line.strip()
        for line in docling_markdown.splitlines()
        if line.strip().startswith(('•', '- '))
    ]
    print("Bullet lines extracted from Docling:", bullet_lines)  # DEBUG

    # Keep only bullets not already in bs4_body
    unique_bullets = [
        bullet for bullet in bullet_lines
        if bullet not in bs4_body
    ]
    if not unique_bullets:
        return bs4_body

    # Prepend them at the top with <br> spacing
    bullet_block = '<br><br>' + '<br><br>'.join(unique_bullets) + '<br><br>'
    return bullet_block + bs4_body

# Hybrid parser that works with in-memory HTML content
def hybrid_parser_from_content(filename, html_content, original_file_path=None):
    # Use BeautifulSoup to extract clean body and titles
    bs4_data = parse_with_bs4(html_content)

    # Save HTML to a temporary file to pass to Docling
    temp_path = f"/tmp/{filename}"
    with open(temp_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

    # Use Docling to extract structured markdown
    docling_data = parse_with_docling(temp_path)

    # Extract source path from original file path if provided
    source_path = filename  # Default fallback
    if original_file_path:
        # Extract relative path from HKNews onwards
        path_parts = Path(original_file_path).parts
        if 'HKNews' in path_parts:
            hknews_index = path_parts.index('HKNews')
            # Get category/year/month/filename structure
            relative_parts = path_parts[hknews_index + 1:]
            source_path = '/'.join(relative_parts)

    return {
        'filename': filename,
        'source_path': source_path,
        'bs4_body': bs4_data['body'],
        'bs4_titles': bs4_data['titles'],
        'bs4_paragraphs': bs4_data['paragraphs'],
        'bs4_structured_content': bs4_data['structured_content'],
        'docling_markdown': docling_data['body']
    }

def chunk_docling_markdown(markdown_text):
    """Split docling markdown into sections using ## headers.
    Fallback to one generic chunk if headers are missing or irrelevant.
    """
    chunks = []
    current = {"title": None, "text": ""}
    for line in markdown_text.splitlines():
        if line.startswith("## "):
            if current["title"] or current["text"].strip():
                chunks.append(current)
            current = {"title": line[3:].strip(), "text": ""}
        else:
            current["text"] += line + "\n"

    if current["title"] or current["text"].strip():
        chunks.append(current)

    # If only footers are present, ignore them and use a fallback chunk
    footer_keywords = ["staffnet", "newsletter", "kontakt", "about"]
    only_footers = all(any(kw in (c["title"] or "").lower() for kw in footer_keywords) for c in chunks)

    if len(chunks) < 2 or only_footers:
        return [{"title": "Main article", "text": ""}]
    else:
        return chunks

def distribute_structured_content(structured_content, docling_chunks):
    """Distribute structured BS4 content (title-paragraph pairs) into Docling-defined chunks"""
    n = len(docling_chunks)
    m = len(structured_content)
    
    if n == 0:
        return docling_chunks
    
    # If we have more structured content pieces than docling chunks, distribute evenly
    if m > n:
        items_per_chunk = max(1, m // n)
        
        for i, chunk in enumerate(docling_chunks):
            start = i * items_per_chunk
            end = m if i == n - 1 else (i + 1) * items_per_chunk
            
            # Combine paragraphs from multiple structured content pieces
            combined_paragraphs = []
            combined_titles = []
            
            for j in range(start, min(end, m)):
                content_piece = structured_content[j]
                combined_titles.append(content_piece['title'])
                combined_paragraphs.extend(content_piece['paragraphs'])
            
            # Update chunk title if we have specific titles
            if combined_titles and combined_titles[0] != "Main article":
                chunk["title"] = combined_titles[0]
            
            chunk["body"] = "\n\n" + "\n\n".join(combined_paragraphs)
            chunk["source_titles"] = combined_titles  # Keep track of source titles
    
    # If we have fewer or equal structured content pieces, map one-to-one
    else:
        for i, chunk in enumerate(docling_chunks):
            if i < len(structured_content):
                content_piece = structured_content[i]
                if content_piece['title'] != "Main article":
                    chunk["title"] = content_piece['title']
                chunk["body"] = "\n\n" + "\n\n".join(content_piece['paragraphs'])
                chunk["source_titles"] = [content_piece['title']]
            else:
                chunk["body"] = ""
                chunk["source_titles"] = []
    
    return docling_chunks

# Note: You'll need to define 'examples' dictionary with your HTML content
# hybrid_results = []
# for filename, html_content in examples.items():
#     result = hybrid_parser_from_content(filename, html_content)
#     hybrid_results.append(result)

# Apply hybrid merging (updated to use structured content)
def process_hybrid_results(hybrid_results):
    final_chunks_per_file = []
    
    for document in hybrid_results:
        # Step 1: Create structural chunks from the markdown document
        docling_chunks = chunk_docling_markdown(document['docling_markdown'])
        
        # Step 2: Distribute the structured BS4 content into the markdown structure
        content_chunks = distribute_structured_content(document['bs4_structured_content'], docling_chunks)
        
        # Step 3: Inject bullet points into the first chunk's body (only if needed)
        if content_chunks:
            content_chunks[0]["body"] = inject_docling_bullets(
                bs4_body=content_chunks[0]["body"],
                docling_markdown=document['docling_markdown']
            )
        
        # FIXED: Use the source_path that was already correctly extracted 
        # in hybrid_parser_from_content, don't override it
        source_path = document.get('source_path', document['filename'])
        
        # Add the processed document to results
        final_chunks_per_file.append({
            'filename': document['filename'],
            'source_path': source_path,  # Use the original extracted path
            'chunks': content_chunks
        })
    
    return final_chunks_per_file

# Updated output directory
def save_parsed_files(final_chunks_per_file):
    output_dir = "/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/parsed_markdown_v2"
    os.makedirs(output_dir, exist_ok=True)
    
    for f in final_chunks_per_file:
        filename = f"{f['filename'].replace('.html', '')}.md"
        filepath = os.path.join(output_dir, filename)
        with open(filepath, "w", encoding="utf-8") as out:
            # Write header
            out.write(f"# {f['filename']}\n\n")
            
            # Write source line with full path structure
            source_path = f.get('source_path', f['filename'])
            out.write(f"**Source:** {source_path}\n\n")
            
            # Write content chunks
            for c in f['chunks']:
                out.write(f"## {c['title']}\n\n")
                paragraphs = c['body'].split('<br><br>')  
                for p in paragraphs:
                    clean = p.strip()
                    if clean:
                        out.write(clean + "\n\n")  # Force paragraph block



# Example usage (uncomment and modify as needed):
# When processing HTML files, pass the original file path
base_folder = '/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/HKNews'
#html_files_content = read_all_html_files(base_folder)

# Get file paths with metadata to preserve original paths
html_data_with_metadata = get_html_files_with_metadata(base_folder)

hybrid_results = []
for filename, data in html_data_with_metadata.items():
    result = hybrid_parser_from_content(
        filename=filename, 
        html_content=data['content'],
        original_file_path=data['metadata']['full_path']  # Pass original path
    )
    hybrid_results.append(result)
    
final_chunks_per_file = process_hybrid_results(hybrid_results)
save_parsed_files(final_chunks_per_file)

Found 4390 HTML files
Bullet lines extracted from Docling: []
Bullet lines extracted from Docling: []
Bullet lines extracted from Docling: []
Bullet lines extracted from Docling: []
Bullet lines extracted from Docling: []
Bullet lines extracted from Docling: []
Bullet lines extracted from Docling: []
Bullet lines extracted from Docling: []
Bullet lines extracted from Docling: []
Bullet lines extracted from Docling: ['- Corel PaintShop ProX6 for Windows in several languages, free of charge', '- RealVNC 5.0.5 for Windows, Mac and Linux/Unix in English at Fr. 5.-/license/year', '- GIS-Software Erdas Imagine 2013 (V13.0) for Windows in English at Fr. 500.-/license/year', '- IBM SPSS Floating 22 for Windows, Mac and Linux as multilingual package, free of charge', '- IBM SPSS Node 22 for Windows, Mac and Linux as multilingual package at Fr. 50.-/license/year', '- EndNote X7 for Mac OS X in English at Fr. 30.-/license', '- Parallels Desktop Version 8 for Mac OS X in English at Fr. 35.-/licens

# metadata enrichment

In [ ]:

import os
import json
import re
import string
import unicodedata
import logging
import glob
from tqdm import tqdm
from langdetect import detect
from pathlib import Path

## Read

In [12]:
import os
import glob

# Set folder path
folder_path = '/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/parsed_markdown_v2'

# Find all markdown files
md_files = glob.glob(os.path.join(folder_path, '*.md'))

# Check if any markdown files exist
if md_files:
    sample_md = md_files[0]  # Pick the first file
    print(f"Sample file: {sample_md}")
    
    # Read and display content
    with open(sample_md, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Print a preview
    print("\n--- Content Preview ---\n")
    print(content[:1000])  # Show first 1000 characters
else:
    print("No markdown files found.")

Sample file: /Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/parsed_markdown_v2/vigilin-the-lock-keeper.md

--- Content Preview ---

# vigilin-the-lock-keeper.html

**Source:** vigilin-the-lock-keeper.html

## From fatty liver to heart attack

Anyone attending Munich’s famous Oktoberfest will know it can leave physical traces; fatty foods and plenty of alcohol cause the liver to work overtime. This organ stores a portion of any fat consumed (and also converts alcohol to fat), but releases it again once the revelry is over.

However, should the excesses continue – and if they are not offset by exercise and sport – the person becomes overweight and diabetic, leading to a condition known as fatty liver. If corrective action is taken in time, the liver can usually recover completely from its fatty degeneration. In severe cases the organ becomes inflamed – a condition that is almost impossible to treat.

## From fatty liver to heart attack

Fatty liver is also associ

# Processing

## Basic

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def basic_text_cleaning(text):
    """Clean and normalize text"""
    text = unicodedata.normalize('NFKC', text)
    text = text.strip()
    return text

def detect_language(text):
    """Detect language of text"""
    if not text:
        return "unknown"
    try:
        sample = text[:1000]
        language = detect(sample)
        return language
    except:
        return "unknown"



def extract_article_structure(markdown_text):
    """Extract structured content from markdown"""
    article = {
        'original_filename': '',
        'source_path': '',
        'date': '',
        'sections': {},
        'paragraphs': []  # New: store paragraphs with headings
    }
    
    # Extract header (# title)
    header_match = re.search(r'^# (.+?)$', markdown_text, re.MULTILINE)
    if header_match:
        article['original_filename'] = header_match.group(1).strip()
    
    # Extract date and source
    date, source_path = extract_date_from_source(markdown_text)
    if date:
        article['date'] = date
    if source_path:
        article['source_path'] = source_path
    
    lines = markdown_text.split('\n')
    current_section = None
    section_content = []
    
    # Track if we've seen any ## headers
    has_section_headers = any(line.startswith('## ') for line in lines)
    first_content_processed = False
    
    for line in lines:
        # Skip the main header and source line
        if line.startswith('# ') or line.startswith('**Source:**'):
            continue
            
        if line.startswith('## '):
            # Save previous section
            if current_section is not None:
                content = '\n'.join(section_content).strip()
                if content:
                    article['sections'][current_section] = content
                    article['paragraphs'].append({
                        'heading': current_section,
                        'text': basic_text_cleaning(content)
                    })
            
            # Start new section
            current_section = line[3:].strip()
            section_content = []
            
        elif current_section is not None:
            section_content.append(line)
            
        elif not first_content_processed and line.strip() and not line.startswith('#'):
            # Handle content before first ## header
            if not has_section_headers or not any(lines[i].startswith('## ') for i in range(len(lines))):
                # No section headers found, or this is content before first header
                current_section = "Introduction"
                section_content = [line]
                first_content_processed = True
            else:
                # There are section headers, so collect content until first header
                current_section = "Introduction"
                section_content = [line]
                first_content_processed = True
    
    # Don't forget the last section
    if current_section is not None and section_content:
        content = '\n'.join(section_content).strip()
        if content:
            article['sections'][current_section] = content
            article['paragraphs'].append({
                'heading': current_section,
                'text': basic_text_cleaning(content)
            })
    
    return article

def extract_title(article_dict, filename):
    """Extract title from filename, header, or source path"""
    
    def extract_title_from_path(path_string):
        """Extract title from path string with pattern: category_year_month_title-parts"""
        # Remove file extensions
        clean_path = path_string.replace('.html', '').replace('.md', '')
        
        # Pattern to match: category_year_month_title-parts
        # Example: de_news_events_2015_05_ein-pionier-des-schweizer-internets-tritt-ab
        title_pattern = r'.*?_(\d{4})_(\d{2})_(.+)

## Extract date

In [ ]:
def extract_date_from_source(markdown_text):
    """
    Extract date (year-month) from source line
    Example: **Source:** de_news_events/2015/03/la-bella-vita.html -> 2015-03
    """
    source_pattern = r'\*\*Source:\*\*\s*(.+?)(?:\n|$)'
    source_match = re.search(source_pattern, markdown_text)
    
    if source_match:
        source_path = source_match.group(1).strip()
        # Extract year and month from path like: category/year/month/filename
        date_pattern = r'(\d{4})/(\d{2})'
        date_match = re.search(date_pattern, source_path)
        
        if date_match:
            year = date_match.group(1)
            month = date_match.group(2)
            return f"{year}-{month}", source_path
    
    return None, None

## extract named entity

In [ ]:
import os
import json
import re
import spacy
from collections import Counter
import logging
from tqdm import tqdm

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class NamedEntityExtractor:
    def __init__(self):
        """Initialize the NER extractor with German and English models"""
        self.models = {}
        
        # Try to load German model
        try:
            self.models['de'] = spacy.load("de_core_news_sm")
            logger.info("Loaded German spaCy model")
        except OSError:
            logger.warning("German spaCy model not found. Install with: python -m spacy download de_core_news_sm")
        
        # Try to load English model
        try:
            self.models['en'] = spacy.load("en_core_web_sm")
            logger.info("Loaded English spaCy model")
        except OSError:
            logger.warning("English spaCy model not found. Install with: python -m spacy download en_core_web_sm")
        
        # Fallback to any available model
        if not self.models:
            try:
                # Try multilingual model
                self.models['xx'] = spacy.load("xx_core_web_sm")
                logger.info("Loaded multilingual spaCy model as fallback")
            except OSError:
                logger.error("No spaCy models available. Please install at least one model.")
                raise Exception("No spaCy models available")
    
    def get_model(self, language):
        """Get appropriate spaCy model for language"""
        if language in self.models:
            return self.models[language]
        elif 'de' in self.models and language == 'unknown':
            return self.models['de']  # Default to German for unknown
        elif 'en' in self.models:
            return self.models['en']  # Fallback to English
        elif 'xx' in self.models:
            return self.models['xx']  # Multilingual fallback
        else:
            return list(self.models.values())[0]  # Any available model
    
    def clean_text_for_ner(self, text):
        """Clean text for better NER performance"""
        # Remove markdown formatting
        text = re.sub(r'\*\*(.+?)\*\*', r'\1', text)  # Remove bold
        text = re.sub(r'\*(.+?)\*', r'\1', text)      # Remove italic
        text = re.sub(r'`(.+?)`', r'\1', text)        # Remove code
        text = re.sub(r'#+\s*', '', text)             # Remove headers
        text = re.sub(r'\[(.+?)\]\(.+?\)', r'\1', text)  # Remove links, keep text
        
        # Clean up extra whitespace
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()
        
        return text
    
    def extract_entities_from_text(self, text, language='de'):
        """Extract named entities from text"""
        if not text or len(text.strip()) < 10:
            return []
        
        # Get appropriate model
        nlp = self.get_model(language)
        
        # Clean text
        clean_text = self.clean_text_for_ner(text)
        
        # Process with spaCy
        doc = nlp(clean_text)
        
        entities = []
        for ent in doc.ents:
            # Filter out very short entities or numbers only
            if len(ent.text.strip()) > 1 and not ent.text.isdigit():
                entities.append({
                    'text': ent.text.strip(),
                    'label': ent.label_,
                    'start': ent.start_char,
                    'end': ent.end_char,
                    'confidence': float(ent._.get('score', 0.0)) if hasattr(ent._, 'score') else None
                })
        
        return entities
    
    def categorize_entities(self, entities):
        """Categorize entities into simplified categories"""
        categories = {
            'PERSON': [],      # People
            'ORG': [],         # Organizations
            'LOCATION': [],    # Places
            'DATE': [],        # Dates/Times
            'MISC': []         # Miscellaneous
        }
        
        # Mapping from spaCy labels to our categories
        label_mapping = {
            # German model labels
            'PER': 'PERSON',
            'LOC': 'LOCATION', 
            'ORG': 'ORG',
            'MISC': 'MISC',
            
            # English model labels
            'PERSON': 'PERSON',
            'ORG': 'ORG',
            'GPE': 'LOCATION',      # Geopolitical entity
            'LOCATION': 'LOCATION',
            'DATE': 'DATE',
            'TIME': 'DATE',
            'EVENT': 'MISC',
            'FAC': 'LOCATION',      # Facility
            'PRODUCT': 'MISC',
            'WORK_OF_ART': 'MISC',
            'LAW': 'MISC',
            'LANGUAGE': 'MISC',
            'NORP': 'MISC',         # Nationalities or religious groups
        }
        
        for entity in entities:
            label = entity['label']
            category = label_mapping.get(label, 'MISC')
            categories[category].append(entity['text'])
        
        # Remove duplicates and sort
        for category in categories:
            categories[category] = sorted(list(set(categories[category])))
        
        return categories
    
    def extract_entities_from_article(self, article_data):
        """Extract entities from a processed article"""
        all_entities = []
        language = article_data.get('language', 'de')
        
        # Extract from title
        if article_data.get('title'):
            title_entities = self.extract_entities_from_text(article_data['title'], language)
            all_entities.extend(title_entities)
        
        # Extract from each section/paragraph
        if article_data.get('paragraphs'):
            for paragraph in article_data['paragraphs']:
                if paragraph.get('text'):
                    entities = self.extract_entities_from_text(paragraph['text'], language)
                    all_entities.extend(entities)
        
        # Categorize entities
        categorized = self.categorize_entities(all_entities)
        
        # Count entity frequencies
        entity_counts = Counter([ent['text'] for ent in all_entities])
        
        return {
            'entities': all_entities,
            'categorized': categorized,
            'entity_counts': dict(entity_counts.most_common(20)),  # Top 20 most frequent
            'total_entities': len(all_entities),
            'unique_entities': len(set([ent['text'] for ent in all_entities]))
        }

def process_markdown_files_with_ner(folder_path, output_path=None):
    """Process all markdown files and extract named entities"""
    
    # First, read the processed articles (assuming you have them from previous script)
    # If not, you'll need to run the markdown processor first
    
    extractor = NamedEntityExtractor()
    
    # Read markdown files directly if no processed articles available
    import glob
    from pathlib import Path
    
    pattern = os.path.join(folder_path, '*.md')
    markdown_files = glob.glob(pattern)
    
    logger.info(f"Found {len(markdown_files)} markdown files for NER processing")
    
    results = []
    
    for file_path in tqdm(markdown_files, desc="Extracting named entities"):
        try:
            filename = os.path.basename(file_path)
            
            # Read markdown content
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
            
            # Simple extraction for NER (you can integrate with full processor later)
            article_data = {
                'filename': filename,
                'title': '',  # Extract from first line if needed
                'language': 'de',  # Default to German
                'paragraphs': [{'text': content}]  # Use full content for now
            }
            
            # Extract title from content if possible
            lines = content.split('\n')
            for line in lines:
                if line.startswith('# ') and not line.startswith('# *.html'):
                    article_data['title'] = line[2:].strip()
                    break
            
            # Extract named entities
            ner_result = extractor.extract_entities_from_article(article_data)
            
            # Combine with article info
            result = {
                'filename': filename,
                'title': article_data['title'],
                'ner': ner_result
            }
            
            results.append(result)
            
        except Exception as e:
            logger.error(f"Error processing {file_path}: {e}")
    
    # Save results
    if output_path:
        save_ner_results(results, output_path)
    
    return results

def save_ner_results(results, output_path):
    """Save NER results to JSON file"""
    try:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        
        logger.info(f"Saved NER results for {len(results)} files to {output_path}")
        
    except Exception as e:
        logger.error(f"Error saving NER results to {output_path}: {e}")



## enrich with more metadata

In [ ]:
import re
import json
from datetime import datetime
from collections import defaultdict
import spacy
from typing import Dict, List, Set

class RAGMetadataExtractor:
    def __init__(self):
        """Initialize metadata extractor for RAG optimization"""
        self.research_keywords = {
            'climate_change': ['climate change', 'global warming', 'carbon emissions', 'sustainability', 'renewable energy'],
            'medical_research': ['cancer', 'treatment', 'therapy', 'medical', 'health', 'disease', 'diagnosis'],
            'technology': ['AI', 'artificial intelligence', 'machine learning', 'quantum computing', 'blockchain'],
            'engineering': ['materials', 'construction', 'infrastructure', 'mechanical', 'electrical'],
            'biology': ['genetics', 'biotechnology', 'molecular', 'cellular', 'organisms', 'ecology'],
            'physics': ['quantum', 'particle physics', 'theoretical physics', 'experimental'],
            'chemistry': ['chemical', 'molecular', 'synthesis', 'catalysis', 'polymer'],
            'computer_science': ['algorithms', 'software', 'programming', 'data science', 'cybersecurity'],
            'mathematics': ['statistics', 'mathematical', 'computational', 'modeling', 'analysis']
        }
        
        self.institution_roles = [
            'president', 'rector', 'professor', 'dean', 'director', 'chair', 'head',
            'researcher', 'scientist', 'alumnus', 'alumni', 'student', 'graduate'
        ]
        
        self.grant_types = [
            'ERC', 'SNF', 'EU', 'Horizon', 'Marie Curie', 'fellowship', 'grant', 'funding'
        ]
    
    def extract_temporal_metadata(self, article_data):
        """Extract time-related metadata for temporal queries"""
        temporal_info = {
            'publication_date': article_data.get('date', ''),
            'mentioned_years': [],
            'time_periods': [],
            'temporal_context': ''
        }
        
        # Extract all text for temporal analysis
        full_text = f"{article_data.get('title', '')} {' '.join([p.get('text', '') for p in article_data.get('paragraphs', [])])}"
        
        # Find explicit years (1900-2030)
        years = re.findall(r'\b(19[0-9]{2}|20[0-3][0-9])\b', full_text)
        temporal_info['mentioned_years'] = sorted(list(set(years)))
        
        # Find time periods and ranges
        time_ranges = re.findall(r'\b(\d{4})[- ]*(?:to|bis|and|und)[- ]*(\d{4})\b', full_text)
        temporal_info['time_periods'] = time_ranges
        
        # Identify temporal context keywords
        temporal_keywords = ['since', 'until', 'during', 'between', 'from', 'over the years', 'currently', 'recently']
        found_temporal = [kw for kw in temporal_keywords if kw.lower() in full_text.lower()]
        temporal_info['temporal_context'] = ', '.join(found_temporal)
        
        return temporal_info
    
    def extract_people_metadata(self, article_data):
        """Extract person-related metadata with roles and context"""
        people_info = {
            'mentioned_people': [],
            'people_with_roles': {},
            'institutional_roles': [],
            'research_collaborators': []
        }
        
        # Get named entities (assuming NER has been run)
        ner_data = article_data.get('ner', {})
        people = ner_data.get('categorized', {}).get('PERSON', [])
        
        full_text = f"{article_data.get('title', '')} {' '.join([p.get('text', '') for p in article_data.get('paragraphs', [])])}"
        
        for person in people:
            people_info['mentioned_people'].append(person)
            
            # Find roles associated with each person
            person_context = self._extract_person_context(person, full_text)
            if person_context:
                people_info['people_with_roles'][person] = person_context
        
        # Extract institutional roles mentioned
        for role in self.institution_roles:
            if role.lower() in full_text.lower():
                people_info['institutional_roles'].append(role)
        
        return people_info
    
    def _extract_person_context(self, person_name, text):
        """Extract context around a person's mention"""
        context = []
        
        # Look for sentences containing the person
        sentences = re.split(r'[.!?]+', text)
        for sentence in sentences:
            if person_name.lower() in sentence.lower():
                # Check for roles in the same sentence
                for role in self.institution_roles:
                    if role.lower() in sentence.lower():
                        context.append(f"{role}")
                
                # Check for titles (Prof., Dr., etc.)
                titles = re.findall(r'\b(Prof\.?|Dr\.?|Professor)\s+' + re.escape(person_name), sentence, re.IGNORECASE)
                context.extend(titles)
        
        return list(set(context))
    
    def extract_research_metadata(self, article_data):
        """Extract research domain and topic metadata"""
        research_info = {
            'research_domains': [],
            'research_topics': [],
            'grants_funding': [],
            'collaborations': [],
            'projects': []
        }
        
        full_text = f"{article_data.get('title', '')} {' '.join([p.get('text', '') for p in article_data.get('paragraphs', [])])}"
        full_text_lower = full_text.lower()
        
        # Classify research domains
        for domain, keywords in self.research_keywords.items():
            if any(keyword.lower() in full_text_lower for keyword in keywords):
                research_info['research_domains'].append(domain)
        
        # Extract specific research topics (from entities)
        ner_data = article_data.get('ner', {})
        misc_entities = ner_data.get('categorized', {}).get('MISC', [])
        research_info['research_topics'] = [entity for entity in misc_entities if len(entity) > 3]
        
        # Find grants and funding
        for grant_type in self.grant_types:
            if grant_type.lower() in full_text_lower:
                research_info['grants_funding'].append(grant_type)
        
        # Extract project names (capitalized phrases)
        projects = re.findall(r'\b[A-Z][a-zA-Z]*[-\s][A-Z][a-zA-Z]*\b', full_text)
        research_info['projects'] = list(set(projects))
        
        return research_info
    
    def extract_institutional_metadata(self, article_data):
        """Extract ETH-specific and institutional metadata"""
        institutional_info = {
            'departments': [],
            'institutions': [],
            'locations': [],
            'programs': [],
            'events': []
        }
        
        # Get organizations and locations from NER
        ner_data = article_data.get('ner', {})
        orgs = ner_data.get('categorized', {}).get('ORG', [])
        locations = ner_data.get('categorized', {}).get('LOCATION', [])
        
        institutional_info['institutions'] = orgs
        institutional_info['locations'] = locations
        
        full_text = f"{article_data.get('title', '')} {' '.join([p.get('text', '') for p in article_data.get('paragraphs', [])])}"
        
        # Extract department mentions
        dept_patterns = [
            r'\bDepartment of ([A-Za-z\s]+)',
            r'\bInstitute for ([A-Za-z\s]+)',
            r'\bFaculty of ([A-Za-z\s]+)',
            r'\b([A-Za-z\s]+) Department\b'
        ]
        
        for pattern in dept_patterns:
            matches = re.findall(pattern, full_text, re.IGNORECASE)
            institutional_info['departments'].extend(matches)
        
        # Extract program mentions
        program_keywords = ['program', 'programme', 'initiative', 'ETH Plus', 'competition']
        for keyword in program_keywords:
            if keyword.lower() in full_text.lower():
                institutional_info['programs'].append(keyword)
        
        return institutional_info
    
    def extract_content_metadata(self, article_data):
        """Extract content-specific metadata for better semantic search"""
        content_info = {
            'question_types': [],
            'content_categories': [],
            'key_concepts': [],
            'document_structure': {},
            'semantic_tags': []
        }
        
        title = article_data.get('title', '')
        paragraphs = article_data.get('paragraphs', [])
        
        # Analyze question types the content might answer
        question_indicators = {
            'who': ['who', 'person', 'researcher', 'scientist', 'professor'],
            'what': ['what', 'research', 'project', 'technology', 'method'],
            'when': ['when', 'date', 'year', 'time', 'period'],
            'where': ['where', 'location', 'place', 'country', 'university'],
            'how': ['how', 'method', 'process', 'technique', 'approach'],
            'why': ['why', 'reason', 'purpose', 'goal', 'objective']
        }
        
        full_text = f"{title} {' '.join([p.get('text', '') for p in paragraphs])}"
        full_text_lower = full_text.lower()
        
        for q_type, indicators in question_indicators.items():
            if any(indicator in full_text_lower for indicator in indicators):
                content_info['question_types'].append(q_type)
        
        # Document structure analysis
        content_info['document_structure'] = {
            'total_sections': len(paragraphs),
            'section_titles': [p.get('heading', '') for p in paragraphs],
            'has_introduction': any('introduction' in p.get('heading', '').lower() for p in paragraphs),
            'content_length': len(full_text)
        }
        
        # Extract key concepts (frequent important terms)
        ner_data = article_data.get('ner', {})
        entity_counts = ner_data.get('entity_counts', {})
        content_info['key_concepts'] = list(entity_counts.keys())[:10]  # Top 10 most frequent entities
        
        return content_info
    
    def create_comprehensive_metadata(self, article_data):
        """Create comprehensive metadata for RAG optimization"""
        metadata = {
            'basic_info': {
                'filename': article_data.get('filename', ''),
                'title': article_data.get('title', ''),
                'language': article_data.get('language', ''),
                'source_path': article_data.get('source_path', '')
            },
            'temporal': self.extract_temporal_metadata(article_data),
            'people': self.extract_people_metadata(article_data),
            'research': self.extract_research_metadata(article_data),
            'institutional': self.extract_institutional_metadata(article_data),
            'content': self.extract_content_metadata(article_data)
        }
        
        # Add retrieval-specific tags
        metadata['retrieval_tags'] = self._create_retrieval_tags(metadata)
        
        return metadata
    
    def _create_retrieval_tags(self, metadata):
        """Create specific tags for different retrieval methods"""
        tags = {
            'bm25_keywords': [],      # For keyword-based retrieval
            'semantic_concepts': [],   # For dense retrieval
            'graph_relations': []      # For graph-based retrieval
        }
        
        # BM25 keywords (high-value terms for exact matching)
        bm25_terms = []
        bm25_terms.extend(metadata['people']['mentioned_people'])
        bm25_terms.extend(metadata['research']['grants_funding'])
        bm25_terms.extend(metadata['institutional']['institutions'])
        bm25_terms.extend(metadata['temporal']['mentioned_years'])
        tags['bm25_keywords'] = list(set(bm25_terms))
        
        # Semantic concepts (for embedding-based retrieval)
        semantic_terms = []
        semantic_terms.extend(metadata['research']['research_domains'])
        semantic_terms.extend(metadata['research']['research_topics'])
        semantic_terms.extend(metadata['content']['question_types'])
        tags['semantic_concepts'] = list(set(semantic_terms))
        
        # Graph relations (entity relationships)
        relations = []
        for person, roles in metadata['people']['people_with_roles'].items():
            for role in roles:
                relations.append(f"{person}_has_role_{role}")
        
        for domain in metadata['research']['research_domains']:
            relations.append(f"research_domain_{domain}")
        
        tags['graph_relations'] = relations
        
        return tags

def process_articles_with_rag_metadata(processed_articles):
    """Add RAG-optimized metadata to processed articles"""
    extractor = RAGMetadataExtractor()
    
    enhanced_articles = []
    for article in processed_articles:
        try:
            # Create comprehensive metadata
            rag_metadata = extractor.create_comprehensive_metadata(article)
            
            # Add to article
            article['rag_metadata'] = rag_metadata
            enhanced_articles.append(article)
            
        except Exception as e:
            print(f"Error processing metadata for {article.get('filename', 'unknown')}: {e}")
            enhanced_articles.append(article)  # Add without enhanced metadata
    
    return enhanced_articles

def save_rag_enhanced_articles(articles, output_path):
    """Save articles with RAG metadata"""
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(articles, f, indent=2, ensure_ascii=False)
    print(f"Saved {len(articles)} articles with RAG metadata to {output_path}")

# Example usage
if __name__ == "__main__":
    # Assuming you have processed_articles from previous steps
    # processed_articles = load_processed_articles()
    
    # Example of what the enhanced metadata looks like:
    example_metadata_structure = {
        "basic_info": {
            "filename": "article.md",
            "title": "ETH Research on Climate Change",
            "language": "en",
            "source_path": "en/research/2023/climate-study.html"
        },
        "temporal": {
            "publication_date": "2023-05",
            "mentioned_years": ["2020", "2021", "2023"],
            "time_periods": [("2020", "2023")],
            "temporal_context": "since, during, currently"
        },
        "people": {
            "mentioned_people": ["Prof. Schmidt", "Dr. Mueller"],
            "people_with_roles": {"Prof. Schmidt": ["Professor"], "Dr. Mueller": ["researcher"]},
            "institutional_roles": ["professor", "researcher"]
        },
        "research": {
            "research_domains": ["climate_change", "technology"],
            "research_topics": ["carbon emissions", "renewable energy"],
            "grants_funding": ["ERC", "SNF"],
            "projects": ["Climate-Tech"]
        },
        "institutional": {
            "departments": ["Environmental Sciences"],
            "institutions": ["ETH Zurich"],
            "locations": ["Switzerland", "Zurich"],
            "programs": ["ETH Plus"]
        },
        "content": {
            "question_types": ["who", "what", "how"],
            "key_concepts": ["climate change", "research", "technology"],
            "document_structure": {"total_sections": 3, "content_length": 1500}
        },
        "retrieval_tags": {
            "bm25_keywords": ["Prof. Schmidt", "ERC", "ETH Zurich", "2023"],
            "semantic_concepts": ["climate_change", "technology", "who", "what"],
            "graph_relations": ["Prof. Schmidt_has_role_Professor", "research_domain_climate_change"]
        }
    }
    
    print("Example RAG metadata structure:")
    print(json.dumps(example_metadata_structure, indent=2))

In [16]:
%pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 12.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993222 sha256=d429a500631fc5e3ddcbb8512a38203df4d27fde1059a67505fb8a15b379da68
  Stored in directory: /Users/dd/Library/Caches/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect

[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## run main

In [ ]:
# main
processed_articles_v2

In [2]:
import sys
print(sys.executable)

/opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python


In [3]:
!/opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install spacy


[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install --upgrade pip


In [4]:
!/opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m spacy download en_core_web_sm
!/opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m spacy download de_core_news_sm
!/opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m spacy download xx_ent_wiki_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 13.3 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 13.4 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 13.3 MB/s eta 0:00:0000:010:01

[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.2.5_1/libexec/bin/python -m pip install --upgrade pip
✔ Downlo

In [1]:
#!/usr/bin/env python3
"""
Complete Processing Pipeline for Markdown Files
Combines: Markdown processing + NER + RAG metadata extraction
"""

import os
import json
import re
import glob
import spacy
import unicodedata
import logging
from datetime import datetime
from collections import Counter, defaultdict
from tqdm import tqdm
from pathlib import Path

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Try to import langdetect, install if missing
try:
    from langdetect import detect
except ImportError:
    logger.warning("langdetect not installed. Installing...")
    os.system("pip install langdetect")
    from langdetect import detect

class CompleteArticleProcessor:
    def __init__(self):
        """Initialize the complete processing pipeline"""
        self.setup_ner_models()
        self.setup_research_keywords()
        
    def setup_ner_models(self):
        """Setup spaCy NER models"""
        self.ner_models = {}
        
        # Try to load German model
        try:
            self.ner_models['de'] = spacy.load("de_core_news_sm")
            logger.info("Loaded German spaCy model")
        except OSError:
            logger.warning("German spaCy model not found. Install with: python -m spacy download de_core_news_sm")
        
        # Try to load English model
        try:
            self.ner_models['en'] = spacy.load("en_core_web_sm")
            logger.info("Loaded English spaCy model")
        except OSError:
            logger.warning("English spaCy model not found. Install with: python -m spacy download en_core_web_sm")
        
        # Fallback to any available model
        if not self.ner_models:
            try:
                self.ner_models['xx'] = spacy.load("xx_core_web_sm")
                logger.info("Loaded multilingual spaCy model as fallback")
            except OSError:
                logger.error("No spaCy models available. Please install at least one model.")
                raise Exception("No spaCy models available")
    
    def setup_research_keywords(self):
        """Setup research domain classification"""
        self.research_keywords = {
            'climate_change': ['climate change', 'global warming', 'carbon emissions', 'sustainability', 'renewable energy'],
            'medical_research': ['cancer', 'treatment', 'therapy', 'medical', 'health', 'disease', 'diagnosis'],
            'technology': ['AI', 'artificial intelligence', 'machine learning', 'quantum computing', 'blockchain'],
            'engineering': ['materials', 'construction', 'infrastructure', 'mechanical', 'electrical'],
            'biology': ['genetics', 'biotechnology', 'molecular', 'cellular', 'organisms', 'ecology'],
            'physics': ['quantum', 'particle physics', 'theoretical physics', 'experimental'],
            'chemistry': ['chemical', 'molecular', 'synthesis', 'catalysis', 'polymer'],
            'computer_science': ['algorithms', 'software', 'programming', 'data science', 'cybersecurity'],
            'mathematics': ['statistics', 'mathematical', 'computational', 'modeling', 'analysis']
        }
        
        self.institution_roles = [
            'president', 'rector', 'professor', 'dean', 'director', 'chair', 'head',
            'researcher', 'scientist', 'alumnus', 'alumni', 'student', 'graduate'
        ]
        
        self.grant_types = [
            'ERC', 'SNF', 'EU', 'Horizon', 'Marie Curie', 'fellowship', 'grant', 'funding'
        ]

    # ===== BASIC TEXT PROCESSING =====
    
    def basic_text_cleaning(self, text):
        """Clean and normalize text"""
        if not text:
            return ""
        text = unicodedata.normalize('NFKC', text)
        text = text.strip()
        return text

    def detect_language(self, text):
        """Detect language of text"""
        if not text:
            return "unknown"
        try:
            sample = text[:1000]
            language = detect(sample)
            return language
        except:
            return "unknown"

    # ===== DATE AND SOURCE EXTRACTION =====
    
    def extract_date_from_source(self, markdown_text):
        """Extract date (year-month) from source line"""
        source_pattern = r'\*\*Source:\*\*\s*(.+?)(?:\n|$)'
        source_match = re.search(source_pattern, markdown_text)
        
        if source_match:
            source_path = source_match.group(1).strip()
            # Extract year and month from path like: category/year/month/filename
            date_pattern = r'(\d{4})[/_](\d{2})'
            date_match = re.search(date_pattern, source_path)
            
            if date_match:
                year = date_match.group(1)
                month = date_match.group(2)
                return f"{year}-{month}", source_path
        
        return None, None

    # ===== TITLE EXTRACTION =====
    
    def extract_title_from_path(self, path_string):
        """Extract title from path string with pattern: category_year_month_title-parts"""
        if not path_string:
            return ""
            
        # Remove file extensions
        clean_path = path_string.replace('.html', '').replace('.md', '')
        
        # Pattern to match: category_year_month_title-parts
        title_pattern = r'.*?_(\d{4})_(\d{2})_(.+)$'
        match = re.search(title_pattern, clean_path)
        
        if match:
            title_part = match.group(3)
        else:
            # Fallback: try to find title after last underscore
            parts = clean_path.split('_')
            if len(parts) >= 4:
                title_part = '_'.join(parts[3:])
            else:
                title_part = clean_path
        
        # Convert from hyphenated to proper title
        title_part = title_part.replace('-', ' ').replace('_', ' ')
        
        # German-aware title case
        words = title_part.split()
        title_words = []
        
        german_lowercase = {'der', 'die', 'das', 'den', 'dem', 'des', 'ein', 'eine', 'einer', 
                           'eines', 'einem', 'und', 'oder', 'aber', 'von', 'zu', 'mit', 
                           'nach', 'bei', 'in', 'an', 'auf', 'für', 'über', 'unter'}
        
        for i, word in enumerate(words):
            if i == 0:
                title_words.append(word.capitalize())
            elif word.lower() in german_lowercase:
                title_words.append(word.lower())
            else:
                title_words.append(word.capitalize())
        
        return ' '.join(title_words)

    def extract_title(self, article_dict, filename):
        """Extract title from filename, header, or source path"""
        
        # Priority 1: Extract from source path
        if article_dict.get('source_path'):
            title = self.extract_title_from_path(article_dict['source_path'])
            if title and len(title.strip()) > 2:
                return self.basic_text_cleaning(title)
        
        # Priority 2: Extract from original filename in header
        if article_dict.get('original_filename'):
            title = self.extract_title_from_path(article_dict['original_filename'])
            if title and len(title.strip()) > 2:
                return self.basic_text_cleaning(title)
        
        # Priority 3: Extract from current markdown filename
        if filename:
            title = self.extract_title_from_path(filename)
            if title and len(title.strip()) > 2:
                return self.basic_text_cleaning(title)
        
        # Priority 4: Look in content sections as fallback
        title_sections = ["In brief", "Main article", "Title", "Headline", "Titel", "Überschrift", "Kurz gefasst", "Introduction"]
        
        for section_name in title_sections:
            if section_name in article_dict['sections']:
                content = article_dict['sections'][section_name]
                lines = content.split('\n')
                if lines:
                    for line in lines:
                        if line.strip() and len(line.strip()) > 2:
                            return self.basic_text_cleaning(line.strip())
        
        # Final fallback
        clean_filename = filename.replace('.md', '').replace('.html', '')
        words = clean_filename.replace('-', ' ').replace('_', ' ').split()
        title = ' '.join(word.capitalize() for word in words)
        return self.basic_text_cleaning(title)

    # ===== ARTICLE STRUCTURE EXTRACTION =====
    
    def extract_article_structure(self, markdown_text):
        """Extract structured content from markdown"""
        article = {
            'original_filename': '',
            'source_path': '',
            'date': '',
            'sections': {},
            'paragraphs': []
        }
        
        # Extract header (# title)
        header_match = re.search(r'^# (.+?)$', markdown_text, re.MULTILINE)
        if header_match:
            article['original_filename'] = header_match.group(1).strip()
        
        # Extract date and source
        date, source_path = self.extract_date_from_source(markdown_text)
        if date:
            article['date'] = date
        if source_path:
            article['source_path'] = source_path
        
        lines = markdown_text.split('\n')
        current_section = None
        section_content = []
        
        # Track if we've seen any ## headers
        has_section_headers = any(line.startswith('## ') for line in lines)
        first_content_processed = False
        
        for line in lines:
            # Skip the main header and source line
            if line.startswith('# ') or line.startswith('**Source:**'):
                continue
                
            if line.startswith('## '):
                # Save previous section
                if current_section is not None:
                    content = '\n'.join(section_content).strip()
                    if content:
                        article['sections'][current_section] = content
                        article['paragraphs'].append({
                            'heading': current_section,
                            'text': self.basic_text_cleaning(content)
                        })
                
                # Start new section
                current_section = line[3:].strip()
                section_content = []
                
            elif current_section is not None:
                section_content.append(line)
                
            elif not first_content_processed and line.strip() and not line.startswith('#'):
                # Handle content before first ## header
                current_section = "Introduction"
                section_content = [line]
                first_content_processed = True
        
        # Don't forget the last section
        if current_section is not None and section_content:
            content = '\n'.join(section_content).strip()
            if content:
                article['sections'][current_section] = content
                article['paragraphs'].append({
                    'heading': current_section,
                    'text': self.basic_text_cleaning(content)
                })
        
        return article

    # ===== NAMED ENTITY RECOGNITION =====
    
    def get_ner_model(self, language):
        """Get appropriate spaCy model for language"""
        if language in self.ner_models:
            return self.ner_models[language]
        elif 'de' in self.ner_models and language == 'unknown':
            return self.ner_models['de']
        elif 'en' in self.ner_models:
            return self.ner_models['en']
        elif 'xx' in self.ner_models:
            return self.ner_models['xx']
        else:
            return list(self.ner_models.values())[0]
    
    def clean_text_for_ner(self, text):
        """Clean text for better NER performance"""
        if not text:
            return ""
        
        # Remove markdown formatting
        text = re.sub(r'\*\*(.+?)\*\*', r'\1', text)
        text = re.sub(r'\*(.+?)\*', r'\1', text)
        text = re.sub(r'`(.+?)`', r'\1', text)
        text = re.sub(r'#+\s*', '', text)
        text = re.sub(r'\[(.+?)\]\(.+?\)', r'\1', text)
        
        # Clean up extra whitespace
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()
        
        return text
    
    def extract_entities_from_text(self, text, language='de'):
        """Extract named entities from text"""
        if not text or len(text.strip()) < 10:
            return []
        
        nlp = self.get_ner_model(language)
        clean_text = self.clean_text_for_ner(text)
        doc = nlp(clean_text)
        
        entities = []
        for ent in doc.ents:
            if len(ent.text.strip()) > 1 and not ent.text.isdigit():
                entities.append({
                    'text': ent.text.strip(),
                    'label': ent.label_,
                    'start': ent.start_char,
                    'end': ent.end_char
                })
        
        return entities
    
    def categorize_entities(self, entities):
        """Categorize entities into simplified categories"""
        categories = {
            'PERSON': [],
            'ORG': [],
            'LOCATION': [],
            'DATE': [],
            'MISC': []
        }
        
        label_mapping = {
            'PER': 'PERSON', 'LOC': 'LOCATION', 'ORG': 'ORG', 'MISC': 'MISC',
            'PERSON': 'PERSON', 'GPE': 'LOCATION', 'LOCATION': 'LOCATION',
            'DATE': 'DATE', 'TIME': 'DATE', 'EVENT': 'MISC', 'FAC': 'LOCATION',
            'PRODUCT': 'MISC', 'WORK_OF_ART': 'MISC', 'LAW': 'MISC',
            'LANGUAGE': 'MISC', 'NORP': 'MISC'
        }
        
        for entity in entities:
            label = entity['label']
            category = label_mapping.get(label, 'MISC')
            categories[category].append(entity['text'])
        
        for category in categories:
            categories[category] = sorted(list(set(categories[category])))
        
        return categories
    
    def extract_ner_from_article(self, article_data):
        """Extract NER from article"""
        all_entities = []
        language = article_data.get('language', 'de')
        
        # Extract from title
        if article_data.get('title'):
            title_entities = self.extract_entities_from_text(article_data['title'], language)
            all_entities.extend(title_entities)
        
        # Extract from paragraphs
        if article_data.get('paragraphs'):
            for paragraph in article_data['paragraphs']:
                if paragraph.get('text'):
                    entities = self.extract_entities_from_text(paragraph['text'], language)
                    all_entities.extend(entities)
        
        categorized = self.categorize_entities(all_entities)
        entity_counts = Counter([ent['text'] for ent in all_entities])
        
        return {
            'entities': all_entities,
            'categorized': categorized,
            'entity_counts': dict(entity_counts.most_common(20)),
            'total_entities': len(all_entities),
            'unique_entities': len(set([ent['text'] for ent in all_entities]))
        }

    # ===== RAG METADATA EXTRACTION =====
    
    def extract_temporal_metadata(self, article_data):
        """Extract time-related metadata"""
        temporal_info = {
            'publication_date': article_data.get('date', ''),
            'mentioned_years': [],
            'time_periods': [],
            'temporal_context': ''
        }
        
        full_text = f"{article_data.get('title', '')} {' '.join([p.get('text', '') for p in article_data.get('paragraphs', [])])}"
        
        # Find years
        years = re.findall(r'\b(19[0-9]{2}|20[0-3][0-9])\b', full_text)
        temporal_info['mentioned_years'] = sorted(list(set(years)))
        
        # Find time ranges
        time_ranges = re.findall(r'\b(\d{4})[- ]*(?:to|bis|and|und)[- ]*(\d{4})\b', full_text)
        temporal_info['time_periods'] = time_ranges
        
        # Temporal context
        temporal_keywords = ['since', 'until', 'during', 'between', 'from', 'over the years', 'currently', 'recently']
        found_temporal = [kw for kw in temporal_keywords if kw.lower() in full_text.lower()]
        temporal_info['temporal_context'] = ', '.join(found_temporal)
        
        return temporal_info
    
    def extract_people_metadata(self, article_data):
        """Extract person-related metadata"""
        people_info = {
            'mentioned_people': [],
            'people_with_roles': {},
            'institutional_roles': []
        }
        
        ner_data = article_data.get('ner', {})
        people = ner_data.get('categorized', {}).get('PERSON', [])
        full_text = f"{article_data.get('title', '')} {' '.join([p.get('text', '') for p in article_data.get('paragraphs', [])])}"
        
        for person in people:
            people_info['mentioned_people'].append(person)
            person_context = self.extract_person_context(person, full_text)
            if person_context:
                people_info['people_with_roles'][person] = person_context
        
        for role in self.institution_roles:
            if role.lower() in full_text.lower():
                people_info['institutional_roles'].append(role)
        
        return people_info
    
    def extract_person_context(self, person_name, text):
        """Extract context around person mention"""
        context = []
        sentences = re.split(r'[.!?]+', text)
        
        for sentence in sentences:
            if person_name.lower() in sentence.lower():
                for role in self.institution_roles:
                    if role.lower() in sentence.lower():
                        context.append(role)
                
                titles = re.findall(r'\b(Prof\.?|Dr\.?|Professor)\s+' + re.escape(person_name), sentence, re.IGNORECASE)
                context.extend(titles)
        
        return list(set(context))
    
    def extract_research_metadata(self, article_data):
        """Extract research domain metadata"""
        research_info = {
            'research_domains': [],
            'research_topics': [],
            'grants_funding': [],
            'projects': []
        }
        
        full_text = f"{article_data.get('title', '')} {' '.join([p.get('text', '') for p in article_data.get('paragraphs', [])])}"
        full_text_lower = full_text.lower()
        
        # Classify research domains
        for domain, keywords in self.research_keywords.items():
            if any(keyword.lower() in full_text_lower for keyword in keywords):
                research_info['research_domains'].append(domain)
        
        # Extract research topics
        ner_data = article_data.get('ner', {})
        misc_entities = ner_data.get('categorized', {}).get('MISC', [])
        research_info['research_topics'] = [entity for entity in misc_entities if len(entity) > 3]
        
        # Find grants
        for grant_type in self.grant_types:
            if grant_type.lower() in full_text_lower:
                research_info['grants_funding'].append(grant_type)
        
        # Extract project names
        projects = re.findall(r'\b[A-Z][a-zA-Z]*[-\s][A-Z][a-zA-Z]*\b', full_text)
        research_info['projects'] = list(set(projects))
        
        return research_info
    
    def extract_institutional_metadata(self, article_data):
        """Extract institutional metadata"""
        institutional_info = {
            'departments': [],
            'institutions': [],
            'locations': [],
            'programs': []
        }
        
        ner_data = article_data.get('ner', {})
        orgs = ner_data.get('categorized', {}).get('ORG', [])
        locations = ner_data.get('categorized', {}).get('LOCATION', [])
        
        institutional_info['institutions'] = orgs
        institutional_info['locations'] = locations
        
        full_text = f"{article_data.get('title', '')} {' '.join([p.get('text', '') for p in article_data.get('paragraphs', [])])}"
        
        # Extract departments
        dept_patterns = [
            r'\bDepartment of ([A-Za-z\s]+)',
            r'\bInstitute for ([A-Za-z\s]+)',
            r'\bFaculty of ([A-Za-z\s]+)'
        ]
        
        for pattern in dept_patterns:
            matches = re.findall(pattern, full_text, re.IGNORECASE)
            institutional_info['departments'].extend(matches)
        
        # Extract programs
        program_keywords = ['program', 'programme', 'initiative', 'ETH Plus', 'competition']
        for keyword in program_keywords:
            if keyword.lower() in full_text.lower():
                institutional_info['programs'].append(keyword)
        
        return institutional_info
    
    def extract_content_metadata(self, article_data):
        """Extract content-specific metadata"""
        content_info = {
            'question_types': [],
            'key_concepts': [],
            'document_structure': {}
        }
        
        title = article_data.get('title', '')
        paragraphs = article_data.get('paragraphs', [])
        
        question_indicators = {
            'who': ['who', 'person', 'researcher', 'scientist', 'professor'],
            'what': ['what', 'research', 'project', 'technology', 'method'],
            'when': ['when', 'date', 'year', 'time', 'period'],
            'where': ['where', 'location', 'place', 'country', 'university'],
            'how': ['how', 'method', 'process', 'technique', 'approach'],
            'why': ['why', 'reason', 'purpose', 'goal', 'objective']
        }
        
        full_text = f"{title} {' '.join([p.get('text', '') for p in paragraphs])}"
        full_text_lower = full_text.lower()
        
        for q_type, indicators in question_indicators.items():
            if any(indicator in full_text_lower for indicator in indicators):
                content_info['question_types'].append(q_type)
        
        content_info['document_structure'] = {
            'total_sections': len(paragraphs),
            'section_titles': [p.get('heading', '') for p in paragraphs],
            'content_length': len(full_text)
        }
        
        ner_data = article_data.get('ner', {})
        entity_counts = ner_data.get('entity_counts', {})
        content_info['key_concepts'] = list(entity_counts.keys())[:10]
        
        return content_info
    
    def create_retrieval_tags(self, metadata):
        """Create retrieval-specific tags"""
        tags = {
            'bm25_keywords': [],
            'semantic_concepts': [],
            'graph_relations': []
        }
        
        # BM25 keywords
        bm25_terms = []
        bm25_terms.extend(metadata['people']['mentioned_people'])
        bm25_terms.extend(metadata['research']['grants_funding'])
        bm25_terms.extend(metadata['institutional']['institutions'])
        bm25_terms.extend(metadata['temporal']['mentioned_years'])
        tags['bm25_keywords'] = list(set(bm25_terms))
        
        # Semantic concepts
        semantic_terms = []
        semantic_terms.extend(metadata['research']['research_domains'])
        semantic_terms.extend(metadata['research']['research_topics'])
        semantic_terms.extend(metadata['content']['question_types'])
        tags['semantic_concepts'] = list(set(semantic_terms))
        
        # Graph relations
        relations = []
        for person, roles in metadata['people']['people_with_roles'].items():
            for role in roles:
                relations.append(f"{person}_has_role_{role}")
        
        for domain in metadata['research']['research_domains']:
            relations.append(f"research_domain_{domain}")
        
        tags['graph_relations'] = relations
        
        return tags
    
    def create_rag_metadata(self, article_data):
        """Create comprehensive RAG metadata"""
        metadata = {
            'basic_info': {
                'filename': article_data.get('filename', ''),
                'title': article_data.get('title', ''),
                'language': article_data.get('language', ''),
                'source_path': article_data.get('source_path', '')
            },
            'temporal': self.extract_temporal_metadata(article_data),
            'people': self.extract_people_metadata(article_data),
            'research': self.extract_research_metadata(article_data),
            'institutional': self.extract_institutional_metadata(article_data),
            'content': self.extract_content_metadata(article_data)
        }
        
        metadata['retrieval_tags'] = self.create_retrieval_tags(metadata)
        return metadata

    # ===== MAIN PROCESSING PIPELINE =====
    
    def process_single_markdown_file(self, file_path):
        """Process a single markdown file through complete pipeline"""
        try:
            filename = os.path.basename(file_path)
            logger.debug(f"Processing {filename}")
            
            # Read markdown content
            with open(file_path, 'r', encoding='utf-8') as f:
                markdown_content = f.read()
            
            # Step 1: Extract article structure
            article = self.extract_article_structure(markdown_content)
            
            # Step 2: Extract title
            title = self.extract_title(article, filename)
            
            # Step 3: Detect language
            all_text = ' '.join([section for section in article['sections'].values()])
            language = self.detect_language(all_text)
            
            # Step 4: Create base article data
            article_data = {
                'filename': filename,
                'original_filename': article['original_filename'],
                'source_path': article['source_path'],
                'date': article['date'],
                'title': title,
                'language': language,
                'sections': article['sections'],
                'paragraphs': article['paragraphs'],
                'metadata': {
                    'total_sections': len(article['sections']),
                    'total_characters': len(all_text),
                    'has_date': bool(article['date']),
                    'has_source': bool(article['source_path'])
                }
            }
            
            # Step 5: Extract named entities
            ner_result = self.extract_ner_from_article(article_data)
            article_data['ner'] = ner_result
            
            # Step 6: Extract RAG metadata
            rag_metadata = self.create_rag_metadata(article_data)
            article_data['rag_metadata'] = rag_metadata
            
            return article_data
            
        except Exception as e:
            logger.error(f"Error processing {file_path}: {e}")
            return None
    
    def process_all_markdown_files(self, input_folder, output_folder):
        """Process all markdown files in folder"""
        
        # Find all markdown files
        pattern = os.path.join(input_folder, '*.md')
        markdown_files = glob.glob(pattern)
        
        if not markdown_files:
            logger.warning(f"No markdown files found in {input_folder}")
            return []
        
        logger.info(f"Found {len(markdown_files)} markdown files to process")
        
        processed_articles = []
        failed_files = []
        
        # Process each file
        for file_path in tqdm(markdown_files, desc="Processing markdown files"):
            result = self.process_single_markdown_file(file_path)
            if result:
                processed_articles.append(result)
            else:
                failed_files.append(file_path)
        
        logger.info(f"Successfully processed: {len(processed_articles)} files")
        if failed_files:
            logger.warning(f"Failed to process: {len(failed_files)} files")
        
        # Save results
        self.save_processed_articles(processed_articles, output_folder)
        
        return processed_articles
    
    def save_processed_articles(self, articles, output_folder):
        """Save processed articles to JSON files"""
        try:
            os.makedirs(output_folder, exist_ok=True)
            
            # Save individual files
            for article in articles:
                filename = article['filename'].replace('.md', '.json')
                output_path = os.path.join(output_folder, filename)
                
                with open(output_path, 'w', encoding='utf-8') as f:
                    json.dump(article, f, indent=2, ensure_ascii=False)
            
            # Save combined file
            combined_path = os.path.join(output_folder, 'all_processed_articles.json')
            with open(combined_path, 'w', encoding='utf-8') as f:
                json.dump(articles, f, indent=2, ensure_ascii=False)
            
            logger.info(f"Saved {len(articles)} processed articles to {output_folder}")
            logger.info(f"Individual files: {len(articles)} JSON files")
            logger.info(f"Combined file: all_processed_articles.json")
            
        except Exception as e:
            logger.error(f"Error saving to {output_folder}: {e}")
    
    def print_processing_statistics(self, articles):
        """Print statistics about processed articles"""
        if not articles:
            return
        
        total_files = len(articles)
        total_entities = sum(a.get('ner', {}).get('total_entities', 0) for a in articles)
        
        # Language distribution
        languages = {}
        for article in articles:
            lang = article.get('language', 'unknown')
            languages[lang] = languages.get(lang, 0) + 1
        
        # Date distribution
        dates = {}
        for article in articles:
            date = article.get('date', 'unknown')
            if date and date != 'unknown':
                year = date.split('-')[0]
                dates[year] = dates.get(year, 0) + 1
        
        # Research domains
        domains = defaultdict(int)
        for article in articles:
            research_domains = article.get('rag_metadata', {}).get('research', {}).get('research_domains', [])
            for domain in research_domains:
                domains[domain] += 1
        
        print("\n" + "="*60)
        print("COMPLETE PROCESSING STATISTICS")
        print("="*60)
        print(f"Total files processed: {total_files}")
        print(f"Total named entities: {total_entities}")
        print(f"Average entities per file: {total_entities/total_files:.1f}")
        
        print(f"\nLanguage distribution:")
        for lang, count in languages.items():
            print(f"  {lang}: {count}")
        
        print(f"\nYear distribution:")
        for year, count in sorted(dates.items()):
            print(f"  {year}: {count}")
        
        print(f"\nTop research domains:")
        for domain, count in sorted(domains.items(), key=lambda x: x[1], reverse=True)[:5]:
            print(f"  {domain}: {count}")
        
        # Show sample
        if articles:
            sample = articles[0]
            print(f"\nSample article: {sample.get('filename', 'N/A')}")
            print(f"  Title: {sample.get('title', 'N/A')}")
            print(f"  Language: {sample.get('language', 'N/A')}")
            print(f"  Date: {sample.get('date', 'N/A')}")
            print(f"  Entities: {sample.get('ner', {}).get('total_entities', 0)}")
            
            rag_meta = sample.get('rag_metadata', {})
            print(f"  Research domains: {rag_meta.get('research', {}).get('research_domains', [])}")
            print(f"  BM25 keywords: {len(rag_meta.get('retrieval_tags', {}).get('bm25_keywords', []))}")

def main():
    """Main execution function"""
    
    # Configuration
    input_folder = '/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/parsed_markdown_v2'
    output_folder = '/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/processed_articles_v2'
    
    # Check input folder exists
    if not os.path.exists(input_folder):
        logger.error(f"Input folder does not exist: {input_folder}")
        return
    
    # Initialize processor
    logger.info("Initializing complete article processor...")
    processor = CompleteArticleProcessor()
    
    # Process all files
    logger.info("Starting complete processing pipeline...")
    processed_articles = processor.process_all_markdown_files(input_folder, output_folder)
    
    # Print statistics
    processor.print_processing_statistics(processed_articles)
    
    logger.info("Processing complete!")

if __name__ == "__main__":
    main()

2025-05-22 18:55:58,449 - INFO - Initializing complete article processor...
2025-05-22 18:55:58,975 - INFO - Loaded German spaCy model
2025-05-22 18:55:59,322 - INFO - Loaded English spaCy model
2025-05-22 18:55:59,323 - INFO - Starting complete processing pipeline...
2025-05-22 18:55:59,344 - INFO - Found 4388 markdown files to process
Processing markdown files: 100%|████████████| 4388/4388 [07:14<00:00, 10.11it/s]
2025-05-22 19:03:13,362 - INFO - Successfully processed: 4388 files
2025-05-22 19:03:16,464 - INFO - Saved 4388 processed articles to /Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/processed_articles_v2
2025-05-22 19:03:16,464 - INFO - Individual files: 4388 JSON files
2025-05-22 19:03:16,464 - INFO - Combined file: all_processed_articles.json
2025-05-22 19:03:16,471 - INFO - Processing complete!



COMPLETE PROCESSING STATISTICS
Total files processed: 4388
Total named entities: 164478
Average entities per file: 37.5

Language distribution:
  en: 2042
  de: 2270
  unknown: 70
  fr: 3
  sv: 1
  it: 2

Year distribution:
  2013: 151
  2014: 415
  2015: 439
  2016: 422
  2017: 273
  2018: 371
  2019: 384
  2020: 438
  2021: 448
  2022: 469
  2023: 340
  2024: 238

Top research domains:
  technology: 2924
  engineering: 890
  medical_research: 696
  chemistry: 497
  mathematics: 449

Sample article: vigilin-the-lock-keeper.md
  Title: En News Events/2016/09/vigilin The Lock Keeper
  Language: en
  Date: 2016-09
  Entities: 36
  Research domains: ['medical_research', 'technology', 'engineering', 'biology', 'chemistry']
  BM25 keywords: 21


## Clean json files 
- delete metadata and sections

In [3]:
import json
import os
import re
from pathlib import Path

def clean_title(title):
    """
    Remove file path components from title, keeping only the actual article title.
    Example: "De News Events/2015/03/la Bella Vita" -> "la Bella Vita"
    """
    if not title:
        return title
    
    # Split by '/' and take the last part (actual title)
    parts = title.split('/')
    actual_title = parts[-1] if parts else title
    
    # Clean up any remaining artifacts
    actual_title = actual_title.strip()
    
    # Remove file extensions if present
    actual_title = re.sub(r'\.(html?|txt|md)$', '', actual_title, flags=re.IGNORECASE)
    
    return actual_title

def clean_json_file(file_path):
    """
    Clean a single JSON file by removing unwanted fields and cleaning the title.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Clean the title if it exists in root
        if 'root' in data and 'title' in data['root']:
            data['root']['title'] = clean_title(data['root']['title'])
        
        # Remove sections field if it exists
        if 'sections' in data:
            del data['sections']
        
        # Remove metadata if it exists
        if 'metadata' in data:
            del data['metadata']
        
        # Write the cleaned data back to the same file
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        
        return True, f"Successfully cleaned {file_path}"
    
    except Exception as e:
        return False, f"Error processing {file_path}: {str(e)}"

def clean_json_directory(directory_path, recursive=True):
    """
    Clean all JSON files in a directory.
    
    Args:
        directory_path: Path to the directory containing JSON files
        recursive: Whether to search subdirectories recursively
    """
    directory = Path(directory_path)
    
    if not directory.exists():
        print(f"Directory {directory_path} does not exist!")
        return
    
    # Find all JSON files
    if recursive:
        json_files = list(directory.rglob("*.json"))
    else:
        json_files = list(directory.glob("*.json"))
    
    if not json_files:
        print(f"No JSON files found in {directory_path}")
        return
    
    print(f"Found {len(json_files)} JSON files to process...")
    
    success_count = 0
    error_count = 0
    
    for json_file in json_files:
        success, message = clean_json_file(json_file)
        if success:
            success_count += 1
            print(f"✓ {message}")
        else:
            error_count += 1
            print(f"✗ {message}")
    
    print(f"\nProcessing complete:")
    print(f"✓ Successfully processed: {success_count} files")
    print(f"✗ Errors: {error_count} files")

def preview_changes(file_path):
    """
    Preview what changes would be made to a JSON file without actually modifying it.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f"\nPreview for: {file_path}")
        print("=" * 50)
        
        # Show title change
        if 'root' in data and 'title' in data['root']:
            original_title = data['root']['title']
            cleaned_title = clean_title(original_title)
            print(f"Title change:")
            print(f"  Before: {original_title}")
            print(f"  After:  {cleaned_title}")
        
        # Show fields that would be removed
        fields_to_remove = []
        if 'sections' in data:
            fields_to_remove.append('sections')
        
        if 'metadata' in data:
            unwanted_fields = ['total_sections', 'total_characters', 'has_data', 'has_source']
            for field in unwanted_fields:
                if field in data['metadata']:
                    fields_to_remove.append(f'metadata.{field}')
        
        if fields_to_remove:
            print(f"\nFields to be removed:")
            for field in fields_to_remove:
                print(f"  - {field}")
        else:
            print(f"\nNo fields to remove.")
        
        # Show what would remain
        remaining_keys = [key for key in data.keys() if key != 'sections']
        print(f"\nRemaining top-level keys: {remaining_keys}")
        
    except Exception as e:
        print(f"Error previewing {file_path}: {str(e)}")

# Example usage:
if __name__ == "__main__":
    # Update this path to point to your JSON files directory
    json_directory = "/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/processed_articles_v2"
    
    # Preview changes on a single file first (optional)
    # preview_changes("/path/to/single/file.json")
    
    # Clean all JSON files in the directory
    clean_json_directory(json_directory, recursive=True)
    
    # If you want to clean just a single file:
    # success, message = clean_json_file("/path/to/single/file.json")
    # print(message)

Found 4391 JSON files to process...
✓ Successfully cleaned /Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/processed_articles_v2/sunbathing-meteoroids.json
✓ Successfully cleaned /Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/processed_articles_v2/de_news_events_2016_02_radical-cairo-in-shenzhen.json
✓ Successfully cleaned /Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/processed_articles_v2/die-eth-zuerich-nimmt-kurs-auf-netto-null.json
✓ Successfully cleaned /Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/processed_articles_v2/how-zurich-has-to-change-its-roads-to-have-more-e-bikes-than-cars.json
✓ Successfully cleaned /Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/processed_articles_v2/wer-in-singapur-forschen-will-soll-sich-jetzt-melden.json
✓ Successfully cleaned /Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/processed_articles_v2/recov

In [ ]:
import json
from pathlib import Path

def clean_title_simple(title):
    """Extract just the actual title from path-like strings."""
    if not title:
        return title
    
    # Take everything after the last slash
    if '/' in title:
        cleaned = title.split('/')[-1]
    else:
        cleaned = title
    
    # Replace underscores and hyphens with spaces
    cleaned = cleaned.replace('_', ' ').replace('-', ' ')
    
    # Clean up multiple spaces and strip
    cleaned = ' '.join(cleaned.split()).strip()
    
    # Capitalize properly
    cleaned = cleaned.title()
    
    return cleaned

def fix_titles_correct(directory_path):
    """Fix titles in JSON files - looking at the correct location."""
    directory = Path(directory_path)
    json_files = list(directory.rglob("*.json"))
    
    print(f"Found {len(json_files)} JSON files")
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Check if title exists directly at root level
            if 'title' in data:
                original = data['title']
                cleaned = clean_title_simple(original)
                
                if original != cleaned:
                    data['title'] = cleaned
                    
                    with open(json_file, 'w', encoding='utf-8') as f:
                        json.dump(data, f, ensure_ascii=False, indent=2)
                    
                    print(f"✓ {json_file.name}")
                    print(f"  Before: '{original}'")
                    print(f"  After:  '{cleaned}'")
                else:
                    print(f"- {json_file.name}: No change needed")
            else:
                print(f"- {json_file.name}: No title field found")
        
        except Exception as e:
            print(f"✗ Error with {json_file}: {e}")

# Usage
if __name__ == "__main__":
    # Update this path
    json_directory = "/Users/dd/Dd/HSLU/4. Semester/2. Advanced GenAI/news-qa-ethz1/notebooks/processed_articles_v2"
    fix_titles_correct(json_directory)